
# Saudi Master Data Generator — Self-contained Notebook

This notebook implements the uploaded Saudi master-data generation specification. It creates these DataFrames:

- `territory_df`
- `salesperson_df`
- `van_df`
- `customer_df`
- `holiday_df`
- `config_df`
- `rfm_scores_df`

It also validates foreign keys and business rules, then writes CSV outputs plus `data_quality_report.json`.


In [21]:

# Self-contained Saudi master/reference data generator
# Run this notebook top-to-bottom. It writes outputs into ./saudi_master_data_output

import sys
import subprocess
import importlib.util
from pathlib import Path
from datetime import date, datetime, timedelta
import json
import math
import random
import string


def ensure_package(package_name, import_name=None):
    """Install package if missing. Safe for local/Jupyter execution."""
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("pandas")
ensure_package("numpy")
ensure_package("faker")

import numpy as np
import pandas as pd
from faker import Faker

try:
    fake = Faker(["ar_SA", "en_US"])
except Exception:
    fake = Faker("en_US")

# -----------------------------
# Static specification inputs
# -----------------------------

TERRITORIES = [
    {
        "territory_id": "TER_RUH",
        "territory_name": "Riyadh Central",
        "center_lat": 24.7136,
        "center_lng": 46.6753,
        "radius_km": 25,
        "warehouse_lat": 24.5790,
        "warehouse_lng": 46.8237,
        "warehouse_address": "Industrial Area, Riyadh",
    },
    {
        "territory_id": "TER_JED",
        "territory_name": "Jeddah North",
        "center_lat": 21.5433,
        "center_lng": 39.1728,
        "radius_km": 22,
        "warehouse_lat": 21.3429,
        "warehouse_lng": 39.2357,
        "warehouse_address": "Al Khomrah Logistics Area, Jeddah",
    },
    {
        "territory_id": "TER_DMM",
        "territory_name": "Dammam Metro",
        "center_lat": 26.4207,
        "center_lng": 50.0888,
        "radius_km": 20,
        "warehouse_lat": 26.2926,
        "warehouse_lng": 50.1629,
        "warehouse_address": "2nd Industrial City, Dammam",
    },
]

LOCALITIES = {
    "TER_RUH": [
        ("Olaya", 24.7115, 46.6746),
        ("Al Malaz", 24.6676, 46.7351),
        ("Al Sulaymaniyah", 24.7012, 46.7112),
        ("Al Yasmin", 24.8271, 46.6302),
        ("Hittin", 24.7636, 46.6022),
    ],
    "TER_JED": [
        ("Al Rawdah", 21.5656, 39.1652),
        ("Al Hamra", 21.5262, 39.1611),
        ("Al Safa", 21.5854, 39.2181),
        ("Al Salamah", 21.5948, 39.1485),
        ("Al Zahra", 21.6152, 39.1335),
    ],
    "TER_DMM": [
        ("Al Faisaliyah", 26.4282, 50.0786),
        ("Al Shati", 26.4701, 50.1124),
        ("Al Mazruiyah", 26.4481, 50.0962),
        ("Al Badiyah", 26.4021, 50.0587),
        ("Al Nuzha", 26.4337, 50.0433),
    ],
}

CITY_CODES = {"TER_RUH": "RUH", "TER_JED": "JED", "TER_DMM": "DMM"}
PLATE_PREFIXES = {"TER_RUH": "RU", "TER_JED": "JE", "TER_DMM": "DM"}

SAUDI_SALESPERSON_NAMES = [
    "Abdullah Al-Qahtani", "Fahad Al-Otaibi", "Mohammed Al-Harbi",
    "Nasser Al-Dossari", "Khalid Al-Ghamdi", "Saeed Al-Zahrani",
    "Yousef Al-Mutairi", "Majed Al-Shammari", "Ahmed Al-Anazi",
    "Salem Al-Rashidi", "Omar Al-Shehri", "Hassan Al-Yami",
    "Rashid Al-Malki", "Ibrahim Al-Subaie", "Mansour Al-Qahtani",
    "Waleed Al-Harbi", "Bilal Khan", "Imran Ahmed", "Sameer Khan",
    "Nadeem Ali", "Arif Rahman", "Mustafa Hussain",
]

OWNER_NAMES = [
    "Al Rajhi", "Al Othman", "Al Harbi", "Al Qahtani", "Al Ghamdi",
    "Al Zahrani", "Al Dossari", "Al Mutairi", "Al Shammari", "Al Anazi",
    "Al Rashid", "Al Saleh", "Al Malki", "Al Subaie", "Al Shehri",
]

BUSINESS_PREFIXES = [
    "Al Noor", "Al Waha", "Al Baraka", "Al Safa", "Al Madina",
    "Al Qassim", "Al Riyadh", "Al Jazeera", "Al Khaleej", "Al Nada",
    "Al Nakheel", "Al Rawabi", "Al Tazaj", "Al Dana", "Al Manar",
]

SHOP_CATEGORIES = [
    "Grocery", "Mini Market", "Supermarket", "Restaurant", "Cafe", "Bakery",
    "Butchery", "Cold Store", "Hotel Kitchen", "Catering Kitchen",
]

BUSINESS_SUFFIX_BY_CATEGORY = {
    "Grocery": ["Grocery", "Baqala", "Food Store"],
    "Mini Market": ["Mini Market", "Corner Market", "Baqala"],
    "Supermarket": ["Supermarket", "Hyper Mini", "Market"],
    "Restaurant": ["Restaurant", "Kitchen", "Grill"],
    "Cafe": ["Cafe", "Coffee House", "Roastery"],
    "Bakery": ["Bakery", "Sweets & Bakery", "Oven"],
    "Butchery": ["Butchery", "Meat Shop", "Fresh Meat"],
    "Cold Store": ["Cold Store", "Frozen Foods", "Chilled Foods"],
    "Hotel Kitchen": ["Hotel Kitchen", "Hospitality Supplies"],
    "Catering Kitchen": ["Catering Kitchen", "Banquet Kitchen"],
}

VISIT_DAYS = ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday"]
ORDER_WINDOWS = ["Morning", "Midday", "Afternoon"]
LIFECYCLE_STATES = ["Active", "New", "At Risk", "Dormant", "Churned"]
LIFECYCLE_PROBS = [0.65, 0.10, 0.15, 0.08, 0.02]

# -----------------------------
# Utility functions
# -----------------------------

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    Faker.seed(seed)


def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance without external geospatial dependencies."""
    r = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def jitter_location(lat, lng, radius_km=2.5):
    lat_jitter = np.random.normal(0, radius_km / 111 / 2)
    lng_jitter = np.random.normal(0, radius_km / (111 * np.cos(np.radians(lat))) / 2)
    return float(lat + lat_jitter), float(lng + lng_jitter)


def random_date_between(start, end):
    days = (end - start).days
    return start + timedelta(days=random.randint(0, days))


def weighted_choice(values, probs):
    return random.choices(values, weights=probs, k=1)[0]


def generate_plate(prefix):
    return f"{prefix}{random.choice(string.ascii_uppercase)} {random.randint(1000, 9999)}"


def performance_multiplier():
    bucket = random.random()
    if bucket < 0.15:
        return round(random.uniform(1.10, 1.20), 2)
    if bucket < 0.85:
        return round(random.uniform(0.95, 1.08), 2)
    return round(random.uniform(0.85, 0.94), 2)


def cold_truck_required(category):
    probs = {
        "Cold Store": 1.00,
        "Butchery": 1.00,
        "Hotel Kitchen": 0.80,
        "Catering Kitchen": 0.75,
        "Restaurant": 0.60,
        "Supermarket": 0.45,
        "Bakery": 0.25,
        "Cafe": 0.20,
        "Grocery": 0.12,
        "Mini Market": 0.12,
    }
    return random.random() < probs[category]


def choose_payment_type(volume_tier):
    credit_prob = {"HIGH": 0.75, "MED": 0.45, "LOW": 0.20}[volume_tier]
    return "credit" if random.random() < credit_prob else "cash"


def generate_credit(volume_tier, payment_type, lifecycle_state):
    if payment_type == "cash":
        return 0.0, 0.0
    limit_ranges = {
        "HIGH": (30000, 120000),
        "MED": (10000, 45000),
        "LOW": (2000, 15000),
    }
    credit_limit = round(random.uniform(*limit_ranges[volume_tier]), 2)
    if lifecycle_state in ["At Risk", "Dormant"]:
        pct = random.uniform(0.55, 1.10)
    elif lifecycle_state == "Churned":
        pct = random.uniform(0.70, 1.10)
    else:
        pct = random.choices(
            [random.uniform(0.05, 0.35), random.uniform(0.35, 0.70), random.uniform(0.70, 1.10)],
            weights=[0.65, 0.25, 0.10],
            k=1,
        )[0]
    return credit_limit, round(credit_limit * pct, 2)


def generate_shop_name(category, locality, used_names):
    suffix = random.choice(BUSINESS_SUFFIX_BY_CATEGORY[category])
    include_locality = random.random() < 0.28
    templates = [
        "{prefix} {suffix}",
        "{owner} {suffix}",
        "{prefix} Fresh {suffix}",
        "{owner} Trading {suffix}",
    ]
    if include_locality:
        templates.extend(["{locality} {suffix}", "{prefix} {locality} {suffix}"])

    # Keep cold-chain suffixes aligned with cold-chain categories.
    if suffix in {"Frozen Foods", "Cold Store", "Chilled Foods"} and category != "Cold Store":
        suffix = random.choice(BUSINESS_SUFFIX_BY_CATEGORY[category])

    for _ in range(50):
        name = random.choice(templates).format(
            prefix=random.choice(BUSINESS_PREFIXES),
            owner=random.choice(OWNER_NAMES),
            locality=locality,
            suffix=suffix,
        )
        if name not in used_names:
            used_names.add(name)
            return name
    name = f"{locality} {suffix} {random.randint(100, 999)}"
    used_names.add(name)
    return name

# -----------------------------
# Generator functions
# -----------------------------

def generate_territories():
    df = pd.DataFrame(TERRITORIES).copy()
    df["default_salesperson"] = None
    df["default_van"] = None
    return df[
        ["territory_id", "territory_name", "warehouse_lat", "warehouse_lng", "warehouse_address",
         "center_lat", "center_lng", "radius_km", "default_salesperson", "default_van"]
    ]


def generate_salespeople(territory_df, salespeople_per_territory=3):
    names = SAUDI_SALESPERSON_NAMES.copy()
    random.shuffle(names)
    rows = []
    for _, ter in territory_df.iterrows():
        code = CITY_CODES[ter.territory_id]
        for i in range(1, salespeople_per_territory + 1):
            rows.append({
                "sales_id": f"SAL_{code}_{i:03d}",
                "name": names.pop() if names else fake.name(),
                "territory_id": ter.territory_id,
                "working_hours_per_day": 8.0,
                "shift_start_time": "09:00",
                "assigned_van": None,
                "active_status": random.random() < 0.93,
                "performance_multiplier": performance_multiplier(),
            })
    return pd.DataFrame(rows)


def generate_vans(territory_df, salesperson_df, backup_vans_per_territory=1):
    rows = []
    for _, ter in territory_df.iterrows():
        n_sales = int((salesperson_df["territory_id"] == ter.territory_id).sum())
        total_vans = n_sales + backup_vans_per_territory
        code = CITY_CODES[ter.territory_id]
        prefix = PLATE_PREFIXES[ter.territory_id]
        for i in range(1, total_vans + 1):
            rows.append({
                "van_id": f"VAN_{code}_{i:03d}",
                "registration_no": generate_plate(prefix),
                "cold_truck_enabled": random.random() < 0.50,
                "territory_id": ter.territory_id,
                "active_status": random.random() < 0.96,
            })
    return pd.DataFrame(rows)


def assign_vans_to_salespeople(salesperson_df, van_df):
    salesperson_df = salesperson_df.copy()
    for territory_id in salesperson_df["territory_id"].unique():
        sp_idx = salesperson_df.index[salesperson_df["territory_id"] == territory_id].tolist()
        vans = van_df[van_df["territory_id"] == territory_id]["van_id"].tolist()
        for idx, van_id in zip(sp_idx, vans):
            salesperson_df.loc[idx, "assigned_van"] = van_id
    return salesperson_df


def assign_defaults(territory_df, salesperson_df, van_df):
    territory_df = territory_df.copy()
    for idx, ter in territory_df.iterrows():
        territory_id = ter.territory_id
        active_sp = salesperson_df[(salesperson_df.territory_id == territory_id) & (salesperson_df.active_status)]
        sp_pool = active_sp if not active_sp.empty else salesperson_df[salesperson_df.territory_id == territory_id]
        default_sp = sp_pool.sort_values("performance_multiplier", ascending=False).iloc[0]
        default_van = default_sp.assigned_van
        if pd.isna(default_van):
            default_van = van_df[van_df.territory_id == territory_id].iloc[0].van_id
        territory_df.loc[idx, "default_salesperson"] = default_sp.sales_id
        territory_df.loc[idx, "default_van"] = default_van
    return territory_df


def generate_customers(territory_df, customers_per_territory=100):
    rows = []
    today = date.today()
    start_acq = today - timedelta(days=5 * 365)
    end_acq = today - timedelta(days=30)

    for _, ter in territory_df.iterrows():
        territory_id = ter.territory_id
        code = CITY_CODES[territory_id]
        used_names = set()
        tiers = ["HIGH"] * 20 + ["MED"] * 30 + ["LOW"] * 50
        random.shuffle(tiers)

        for i in range(1, customers_per_territory + 1):
            locality, base_lat, base_lng = random.choice(LOCALITIES[territory_id])
            for _ in range(100):
                lat, lng = jitter_location(base_lat, base_lng, radius_km=2.5)
                if haversine_km(lat, lng, ter.center_lat, ter.center_lng) <= ter.radius_km:
                    break
            else:
                lat, lng = base_lat, base_lng

            tier = tiers[i - 1]
            lifecycle = weighted_choice(LIFECYCLE_STATES, LIFECYCLE_PROBS)
            category = random.choice(SHOP_CATEGORIES)
            payment_type = choose_payment_type(tier)
            credit_limit, outstanding_balance = generate_credit(tier, payment_type, lifecycle)
            customer_rating = int(random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.25, 0.35, 0.25], k=1)[0])

            rows.append({
                "customer_id": f"CUS_{code}_{i:04d}",
                "shop_name": generate_shop_name(category, locality, used_names),
                "gps_lat": round(lat, 6),
                "gps_lng": round(lng, 6),
                "locality": locality,
                "territory_id": territory_id,
                "customer_rating": customer_rating,
                "review_rating": round(float(np.clip(np.random.normal(4.0, 0.55), 2.5, 5.0)), 1),
                "shop_category": category,
                "cold_truck_required": cold_truck_required(category),
                "volume_tier": tier,
                "payment_type": payment_type,
                "credit_limit": credit_limit,
                "outstanding_balance": outstanding_balance,
                "lifecycle_state": lifecycle,
                "acquisition_date": random_date_between(start_acq, end_acq),
                "preferred_visit_day": random.choice(VISIT_DAYS),
                "preferred_order_window": random.choice(ORDER_WINDOWS),
            })
    return pd.DataFrame(rows)


def fridays_between(start_date, end_date):
    cur = start_date
    while cur.weekday() != 4:  # Monday=0, Friday=4
        cur += timedelta(days=1)
    while cur <= end_date:
        yield cur
        cur += timedelta(days=7)


def generate_holidays(territory_df, salesperson_df, van_df, months=3):
    rows = []
    today = date.today()
    start = date(today.year, today.month, 1)
    end = start + timedelta(days=months * 31)
    holiday_id = 1

    # Territory Friday closure rows
    for _, ter in territory_df.iterrows():
        for d in fridays_between(start, end):
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": None,
                "van_holiday": None,
                "territory_holiday": ter.territory_id,
                "from_date": d,
                "to_date": d,
                "reason": "Friday closure",
            })
            holiday_id += 1

        # Common Saudi master calendar events. Dates are approximate placeholders for synthetic data.
        for label, offset_days, duration_days in [
            ("Ramadan adjusted operating hours", 10, 29),
            ("Eid Al-Fitr holiday", 40, 4),
            ("Saudi National Day", 145, 1),
        ]:
            from_d = start + timedelta(days=offset_days)
            to_d = from_d + timedelta(days=duration_days - 1)
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": None,
                "van_holiday": None,
                "territory_holiday": ter.territory_id,
                "from_date": from_d,
                "to_date": to_d,
                "reason": label,
            })
            holiday_id += 1

    # Salesperson leave: 1-2 short leaves each
    for _, sp in salesperson_df.iterrows():
        for _ in range(random.randint(1, 2)):
            from_d = random_date_between(start, end)
            duration = random.choice([1, 1, 2, 3])
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": sp.sales_id,
                "van_holiday": None,
                "territory_holiday": None,
                "from_date": from_d,
                "to_date": from_d + timedelta(days=duration - 1),
                "reason": random.choice(["Annual leave", "Sick leave"]),
            })
            holiday_id += 1

    # Van maintenance: 1 maintenance day per van per month
    for _, van in van_df.iterrows():
        for month_offset in range(months):
            month_start = start + timedelta(days=month_offset * 31)
            from_d = month_start + timedelta(days=random.randint(0, 27))
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": None,
                "van_holiday": van.van_id,
                "territory_holiday": None,
                "from_date": from_d,
                "to_date": from_d,
                "reason": "Van maintenance",
            })
            holiday_id += 1

    return pd.DataFrame(rows)


def generate_config():
    values = {
        "avg_speed_kmh": 32,
        "avg_service_time_min": 22,
        "buffer_pct": 0.15,
        "rfm_window_days": 90,
        "route_partial_prob": 0.08,
        "route_cancel_prob": 0.03,
        "traffic_jam_prob": 0.12,
        "credit_outstanding_cap": 0.85,
        "normal_shift_start_time": "09:00",
        "ramadan_shift_start_time": "10:00",
    }
    return pd.DataFrame([{"config_key": k, "config_value": v} for k, v in values.items()])


# ─────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# RFM + Context scoring  (0-1 min-max, equal-weight final score)
# ─────────────────────────────────────────────────────────────────────────────
#
# Final customer score = mean of 7 independently normalised factors:
#
#   1. r_score          – recency   (days since last success, inverted)
#   2. f_score          – frequency (# successful visits in last 30 days)
#   3. m_score          – monetary  (SAR revenue in last 30 days)
#   4. territory_score  – avg successful transaction amount per territory
#   5. locality_score   – avg successful transaction amount per locality
#   6. rating_score     – customer_rating (1-5) normalised to 0-1
#   7. seasonality_score– same-calendar-month buying pattern vs prior years
#
# Every factor is min-max scaled to [0, 1] independently before averaging,
# so no factor can dominate by scale.  Weight per factor = 1/7 ≈ 14.3 %.
# ─────────────────────────────────────────────────────────────────────────────


def _filter_last_30_days(visit_df: pd.DataFrame):
    """Filter visit_df to last 30 days; return (filtered_df, analysis_date).
    analysis_date is today so recency is always relative to the current date.
    """
    analysis_date = pd.Timestamp.today().normalize()
    start_date = analysis_date - pd.Timedelta(days=29)
    mask = pd.to_datetime(visit_df["visit_date"]).between(start_date, analysis_date)
    return visit_df[mask].copy(), analysis_date


def _min_max_scale(series: pd.Series, reverse: bool = False) -> pd.Series:
    """
    Normalise a series to [0, 1].
    reverse=True  →  lower raw value gets higher score (used for recency).
    Flat series (all equal) → every value scores 1.0.
    """
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(1.0, index=series.index)
    if reverse:
        return (hi - series) / (hi - lo)
    return (series - lo) / (hi - lo)


def _scale_within_territory(df: pd.DataFrame,
                             col: str,
                             reverse: bool = False) -> pd.Series:
    """
    Apply _min_max_scale separately inside each territory_id group.
    A customer's score reflects where they stand among their own territory
    peers — not across the whole population.
    Returns a Series aligned to df.index.
    """
    result = pd.Series(index=df.index, dtype=float)
    for _, grp in df.groupby("territory_id"):
        result.loc[grp.index] = _min_max_scale(grp[col], reverse=reverse).values
    return result


def _compute_seasonality_score(customer_df: pd.DataFrame,
                                visit_df: pd.DataFrame) -> pd.Series:
    """
    Seasonality score (0-1) based on same-calendar-month historical visits,
    scaled within each territory so customers are ranked against peers.

    Derivation
    ----------
    1. analysis_month = calendar month of the latest visit date in visit_df.
       (e.g. if today is May 2026, analysis_month = 5)

    2. From the full visit history (all years), isolate visits whose calendar
       month == analysis_month — i.e. May 2025, May 2024, … These reveal how
       active the customer is specifically in that month every year.

    3. Per customer compute two signals:
         sm_success_rate  = successful same-month visits / total same-month visits
         sm_visit_share   = same-month visits / customer's all-time total visits

       • sm_success_rate answers: "when they visit this month, do they buy?"
       • sm_visit_share  answers: "is this their naturally busy month?"

    4. Blend: seasonality_raw = 0.6 * sm_success_rate + 0.4 * sm_visit_share

    5. Min-max scale the blended value within each territory (peers only).

    6. Customers with no same-month history receive the neutral score 0.5.

    Returns
    -------
    pd.Series  indexed by customer_id, values in [0, 1].
    """
    v = visit_df.copy()
    v["visit_date"] = pd.to_datetime(v["visit_date"])
    analysis_month = v["visit_date"].max().month

    same_month = v[v["visit_date"].dt.month == analysis_month]

    sm_agg = (
        same_month.groupby("customer_id")
        .agg(
            sm_visits  =("visit_date",       "count"),
            sm_success =("successful_visit", "sum"),
        )
        .reset_index()
    )
    sm_agg["sm_success_rate"] = sm_agg["sm_success"] / sm_agg["sm_visits"].clip(lower=1)

    total_agg = (
        v.groupby("customer_id")
        .agg(total_visits=("visit_date", "count"))
        .reset_index()
    )

    base = customer_df[["customer_id", "territory_id"]].copy()
    merged = base.merge(sm_agg,    on="customer_id", how="left")
    merged = merged.merge(total_agg, on="customer_id", how="left")

    merged["sm_visits"]       = merged["sm_visits"].fillna(0)
    merged["sm_success_rate"] = merged["sm_success_rate"].fillna(0.0)
    merged["total_visits"]    = merged["total_visits"].fillna(1)
    merged["sm_visit_share"]  = merged["sm_visits"] / merged["total_visits"].clip(lower=1)

    has_history = (merged["sm_visits"] > 0).values
    merged["seasonality_raw"] = 0.6 * merged["sm_success_rate"] + 0.4 * merged["sm_visit_share"]

    # Scale within territory — peers only
    scaled = _scale_within_territory(merged.reset_index(drop=True), "seasonality_raw", reverse=False)
    result = scaled.where(pd.Series(has_history), 0.5)
    result.index = merged["customer_id"].values
    return result  # index = customer_id, values in [0, 1]


def generate_rfm_scores(customer_df: pd.DataFrame,
                         visit_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Compute the final customer score table — all scoring done per territory.

    Every min-max normalisation step compares a customer only against others
    in the same territory, so scores are meaningful peer comparisons.

    Parameters
    ----------
    customer_df : pd.DataFrame
        Must contain: customer_id, locality, territory_id, customer_rating.
    visit_df : pd.DataFrame
        Must contain: customer_id, visit_date, successful_visit,
                      transaction_amount.

    Score architecture & weights
    ─────────────────────────────────────────────────────────────────────
    Component           Weight   Scope     Source
    ──────────────────  ───────  ────────  ──────────────────────────────
    rfm_combined         40 %   territory  mean(r_score, f_score, m_score)
      r_score                              days since last success (inverted)
      f_score                              successful visit count — last 30d
      m_score                              transaction sum — last 30d
    seasonality_score    30 %   territory  same calendar-month buying pattern
    territory_score      10 %   global     territory avg txn vs all territories
    locality_score       10 %   territory  locality avg txn within territory
    rating_score         10 %   territory  customer_rating 1-5

    Segment thresholds on final_customer_score:
        >= 0.70  →  High
        >= 0.40  →  Medium
        <  0.40  →  Low

    customer_rank is within-territory (rank 1 = best in that territory).
    """
    if visit_df is None:
        raise ValueError("visit_df is required to compute RFM scores.")

    visit_df = visit_df.copy()
    visit_df["visit_date"] = pd.to_datetime(visit_df["visit_date"])

    # ── 1. Last-30-day window ────────────────────────────────────────────────
    v30, analysis_date = _filter_last_30_days(visit_df)
    v30_success = v30[v30["successful_visit"] == True].copy()

    # ── 2. Per-customer R/F/M aggregates (last 30 days) ─────────────────────
    rfm_agg = (
        v30_success.groupby("customer_id")
        .agg(
            last_successful_visit=("visit_date",         "max"),
            frequency            =("successful_visit",   "sum"),
            monetary             =("transaction_amount",  "sum"),
        )
        .reset_index()
    )
    rfm_agg["recency"] = (analysis_date - rfm_agg["last_successful_visit"]).dt.days

    # ── 3. Merge onto full customer base; fill customers with no activity ────
    base = customer_df[["customer_id", "locality", "territory_id", "customer_rating"]].copy()
    rfm = base.merge(rfm_agg, on="customer_id", how="left")
    rfm["recency"]               = rfm["recency"].fillna(30)
    rfm["frequency"]             = rfm["frequency"].fillna(0)
    rfm["monetary"]              = rfm["monetary"].fillna(0.0)
    rfm["last_successful_visit"] = rfm["last_successful_visit"].fillna(pd.NaT)

    # ── 4. Factors 1-3: r / f / m — scaled within territory ─────────────────
    rfm["r_score"] = _scale_within_territory(rfm, "recency",   reverse=True)
    rfm["f_score"] = _scale_within_territory(rfm, "frequency", reverse=False)
    rfm["m_score"] = _scale_within_territory(rfm, "monetary",  reverse=False)

    # ── 5. territory_score ──────────────────────────────────────────────────
    # Avg successful transaction per territory, scaled GLOBALLY across all
    # territories — same logic as locality_score but one level up.
    # Every customer in a territory inherits that territory's average, then
    # all three territory averages are min-max scaled against each other so
    # the best-performing territory gets 1.0 and the weakest gets 0.0.
    succ_all = visit_df[visit_df["successful_visit"] == True].copy()
    succ_ter = succ_all.merge(
        customer_df[["customer_id", "territory_id"]], on="customer_id", how="left"
    )
    ter_avg = (
        succ_ter.groupby("territory_id")["transaction_amount"]
        .mean()
        .reset_index()
        .rename(columns={"transaction_amount": "ter_avg_amount"})
    )
    # Globally scale territory averages across all territories
    ter_avg["territory_score"] = _min_max_scale(ter_avg["ter_avg_amount"])
    # Map back to every customer in that territory
    rfm = rfm.merge(ter_avg[["territory_id", "territory_score"]], on="territory_id", how="left")
    rfm["territory_score"] = rfm["territory_score"].fillna(0.0)

    # ── 6. locality_score ───────────────────────────────────────────────────
    # Avg successful transaction per locality, scaled within territory so
    # localities are compared only to peers inside the same territory.
    succ_loc = succ_all.merge(
        customer_df[["customer_id", "locality", "territory_id"]], on="customer_id", how="left"
    )
    loc_avg = (
        succ_loc.groupby(["territory_id", "locality"])["transaction_amount"]
        .mean()
        .reset_index()
        .rename(columns={"transaction_amount": "loc_avg_amount"})
    )
    rfm = rfm.merge(loc_avg, on=["territory_id", "locality"], how="left")
    rfm["loc_avg_amount"] = rfm["loc_avg_amount"].fillna(
        rfm["territory_id"].map(ter_avg.set_index("territory_id")["ter_avg_amount"])
    )
    rfm["locality_score"] = _scale_within_territory(rfm, "loc_avg_amount", reverse=False)

    # ── 7. rating_score — scaled within territory ────────────────────────────
    rfm["rating_score"] = _scale_within_territory(rfm, "customer_rating", reverse=False)

    # ── 8. seasonality_score — scaled within territory ───────────────────────
    seasonality_map          = _compute_seasonality_score(customer_df, visit_df)
    rfm["seasonality_score"] = rfm["customer_id"].map(seasonality_map).fillna(0.5)

    # ── 9. Final customer score ──────────────────────────────────────────────
    #
    #   Weight breakdown:
    #       rfm_combined   40 %  →  mean(r_score, f_score, m_score)
    #       seasonality    30 %  →  seasonality_score
    #       territory      10 %  →  territory_score
    #       locality       10 %  →  locality_score
    #       rating         10 %  →  rating_score
    #
    #   RFM is first collapsed to a single combined score (equal sub-weights
    #   for R, F, M), then that combined score carries 40 % of the total.
    #
    W_RFM = 0.40
    W_SEA = 0.30
    W_TER = 0.10
    W_LOC = 0.10
    W_RAT = 0.10

    rfm["rfm_combined"] = rfm[["r_score", "f_score", "m_score"]].mean(axis=1)

    rfm["final_customer_score"] = (
        W_RFM * rfm["rfm_combined"]
        + W_SEA * rfm["seasonality_score"]
        + W_TER * rfm["territory_score"]
        + W_LOC * rfm["locality_score"]
        + W_RAT * rfm["rating_score"]
    )

    # ── 10. Segment and within-territory rank ────────────────────────────────
    def _assign_segment(score: float) -> str:
        if score >= 0.70:
            return "High"
        if score >= 0.40:
            return "Medium"
        return "Low"

    rfm["rfm_segment_final"] = rfm["final_customer_score"].apply(_assign_segment)

    # Rank within territory: rank 1 = highest scorer in that territory
    rfm["customer_rank"] = (
        rfm.groupby("territory_id")["final_customer_score"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    rfm = rfm.sort_values(
        ["territory_id", "customer_rank"]
    ).reset_index(drop=True)

    return rfm[[
        "customer_id",
        "territory_id",
        # raw R/F/M inputs
        "last_successful_visit", "recency", "frequency", "monetary",
        # component scores (all 0-1)
        "r_score", "f_score", "m_score",
        "rfm_combined",       # mean(r, f, m) — carries 40 % weight
        "seasonality_score",  # 30 %
        "territory_score",    # 10 %  (global across territories)
        "locality_score",     # 10 %  (within territory)
        "rating_score",       # 10 %
        # final
        "final_customer_score",
        "rfm_segment_final",
        "customer_rank",      # rank within territory
    ]]


# -----------------------------
# Validation and reporting
# -----------------------------

def assert_fk(child_df, child_col, parent_df, parent_col):
    missing = set(child_df[child_col].dropna()) - set(parent_df[parent_col])
    assert not missing, f"Broken FK {child_col}: {missing}"


def validate_all(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df):
    # Foreign keys
    assert_fk(customer_df, "territory_id", territory_df, "territory_id")
    assert_fk(salesperson_df, "territory_id", territory_df, "territory_id")
    assert_fk(van_df, "territory_id", territory_df, "territory_id")
    assert_fk(holiday_df, "territory_holiday", territory_df, "territory_id")
    assert_fk(holiday_df, "salesperson_holiday", salesperson_df, "sales_id")
    assert_fk(holiday_df, "van_holiday", van_df, "van_id")
    assert_fk(rfm_scores_df, "customer_id", customer_df, "customer_id")

    # Primary keys
    assert territory_df["territory_id"].is_unique
    assert salesperson_df["sales_id"].is_unique
    assert van_df["van_id"].is_unique
    assert customer_df["customer_id"].is_unique
    assert holiday_df["holiday_id"].is_unique
    assert rfm_scores_df["customer_id"].is_unique

    # Row counts
    assert len(territory_df) == 3
    assert len(customer_df) == 300
    assert 6 <= len(salesperson_df) <= 9
    assert 9 <= len(van_df) <= 12
    assert len(rfm_scores_df) == 300

    # Same territory checks
    van_territory = van_df.set_index("van_id")["territory_id"].to_dict()
    sp_territory = salesperson_df.set_index("sales_id")["territory_id"].to_dict()
    for _, sp in salesperson_df.iterrows():
        assert van_territory[sp.assigned_van] == sp.territory_id
    for _, ter in territory_df.iterrows():
        assert sp_territory[ter.default_salesperson] == ter.territory_id
        assert van_territory[ter.default_van] == ter.territory_id

    # Business rules
    cash = customer_df["payment_type"] == "cash"
    assert (customer_df.loc[cash, "credit_limit"] == 0).all()
    assert (customer_df.loc[cash, "outstanding_balance"] == 0).all()
    credit = customer_df["payment_type"] == "credit"
    assert (customer_df.loc[credit, "credit_limit"] > 0).all()
    assert customer_df.loc[customer_df.shop_category.isin(["Cold Store", "Butchery"]), "cold_truck_required"].all()

    # Customer coordinates inside territory radius
    ter_lookup = territory_df.set_index("territory_id")
    for _, c in customer_df.iterrows():
        ter = ter_lookup.loc[c.territory_id]
        dist = haversine_km(c.gps_lat, c.gps_lng, ter.center_lat, ter.center_lng)
        assert dist <= ter.radius_km + 1e-6, f"Customer outside radius: {c.customer_id} {dist:.2f}km"

    # No duplicate shop names inside territory
    dupes = customer_df.duplicated(subset=["territory_id", "shop_name"]).sum()
    assert dupes == 0, f"Duplicate shop names inside territory: {dupes}"

    # Tier distribution
    tier_counts = customer_df.groupby(["territory_id", "volume_tier"]).size().unstack(fill_value=0)
    for territory_id in territory_df["territory_id"]:
        assert int(tier_counts.loc[territory_id, "HIGH"]) == 20
        assert int(tier_counts.loc[territory_id, "MED"]) == 30
        assert int(tier_counts.loc[territory_id, "LOW"]) == 50

    return True


def data_quality_report(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df):
    report = {
        "row_counts": {
            "territory": len(territory_df),
            "salesperson": len(salesperson_df),
            "van": len(van_df),
            "customer": len(customer_df),
            "holiday": len(holiday_df),
            "config": len(config_df),
            "rfm_scores": len(rfm_scores_df),
        },
        "customer_tiers_by_territory": customer_df.groupby(["territory_id", "volume_tier"]).size().unstack(fill_value=0).to_dict(orient="index"),
        "lifecycle_distribution": customer_df["lifecycle_state"].value_counts(normalize=True).round(3).to_dict(),
        "payment_type_distribution": customer_df["payment_type"].value_counts(normalize=True).round(3).to_dict(),
        "cold_truck_required_share": round(float(customer_df["cold_truck_required"].mean()), 3),
        "rfm_segments": rfm_scores_df["rfm_segment_final"].value_counts().to_dict(),
        "validation_status": "passed",
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }
    return report


# ─────────────────────────────────────────────────────────────────────────────
# Visit Table Generator
# ─────────────────────────────────────────────────────────────────────────────

def _expand_holiday_dates(holiday_df: pd.DataFrame) -> dict:
    """
    Pre-compute blocked date sets from holiday_df for fast O(1) lookup.

    Returns a dict with three keys:

        territory_blocked  : { territory_id  → set of date }
        salesperson_blocked: { sales_id       → set of date }
        van_blocked        : { van_id         → set of date }

    A visit on a date that appears in any of these sets for the matching
    entity is suppressed entirely — the date is simply skipped.
    """
    from datetime import timedelta

    territory_blocked  = {}
    salesperson_blocked = {}
    van_blocked        = {}

    for _, row in holiday_df.iterrows():
        from_d = pd.to_datetime(row["from_date"]).date()
        to_d   = pd.to_datetime(row["to_date"]).date()

        # Expand every date in the [from, to] range
        dates = set()
        cur = from_d
        while cur <= to_d:
            dates.add(cur)
            cur += timedelta(days=1)

        if pd.notna(row["territory_holiday"]) and row["territory_holiday"]:
            tid = row["territory_holiday"]
            territory_blocked.setdefault(tid, set()).update(dates)

        if pd.notna(row["salesperson_holiday"]) and row["salesperson_holiday"]:
            sid = row["salesperson_holiday"]
            salesperson_blocked.setdefault(sid, set()).update(dates)

        if pd.notna(row["van_holiday"]) and row["van_holiday"]:
            vid = row["van_holiday"]
            van_blocked.setdefault(vid, set()).update(dates)

    return {
        "territory_blocked":   territory_blocked,
        "salesperson_blocked": salesperson_blocked,
        "van_blocked":         van_blocked,
    }


def generate_visits(customer_df, salesperson_df, van_df, territory_df,
                    holiday_df=None, start_date=None, end_date=None, seed=42):
    """
    Generate synthetic visit records, respecting holiday_df constraints.

    Holiday blocking rules
    ----------------------
    A visit on date D for (salesperson S, van V, territory T) is skipped if:
        • D is a territory holiday for T  (whole territory closed — e.g. Friday, Eid)
        • D is a salesperson holiday for S (sick leave, annual leave)
        • D is a van holiday for V         (maintenance day)

    All three conditions are checked independently; any one is enough to
    suppress the visit for that date.

    Columns
    -------
    visit_id            : str   – unique identifier e.g. VIS_RUH_000001
    visit_date          : date  – weekday (Sat-Thu), not a Friday, within range
    customer_id         : str   – FK → customer_df
    sales_id            : str   – FK → salesperson_df
    van_id              : str   – FK → van_df
    successful_visit    : bool  – whether a transaction was completed
    transaction_amount  : float – SAR; 0.0 when visit was unsuccessful
    notes               : str   – brief free-text remark
    """
    from datetime import date, timedelta

    random.seed(seed)
    np.random.seed(seed)

    today = date.today()
    if start_date is None:
        start_date = today - timedelta(days=360)
    if end_date is None:
        end_date = today - timedelta(days=1)

    # ── Build holiday blocked-date sets ─────────────────────────────────────
    if holiday_df is not None and len(holiday_df) > 0:
        blocked = _expand_holiday_dates(holiday_df)
    else:
        blocked = {"territory_blocked": {}, "salesperson_blocked": {}, "van_blocked": {}}

    ter_blocked = blocked["territory_blocked"]
    sp_blocked  = blocked["salesperson_blocked"]
    van_blocked = blocked["van_blocked"]

    # ── Build lookup: territory_id → list of (sales_id, van_id) ─────────────
    sp_van = (
        salesperson_df[salesperson_df["active_status"]]
        .merge(
            van_df[van_df["active_status"]][["van_id", "territory_id"]],
            left_on=["territory_id", "assigned_van"],
            right_on=["territory_id", "van_id"],
            how="left",
        )[["territory_id", "sales_id", "assigned_van"]]
        .rename(columns={"assigned_van": "van_id"})
    )
    territory_staff = {}
    for ter_id, grp in sp_van.groupby("territory_id"):
        territory_staff[ter_id] = list(zip(grp["sales_id"], grp["van_id"]))

    success_probs = {"HIGH": 0.92, "MED": 0.82, "LOW": 0.70}
    amount_params = {
        "HIGH": (12000, 4000),
        "MED":  (3500,  1200),
        "LOW":  (700,   300),
    }

    successful_notes = [
        "Order placed successfully.", "Full order confirmed.", "Client satisfied with delivery.",
        "Repeat order; no issues.", "New SKUs added to order.", "Paid on delivery.",
        "Invoice issued.", "Seasonal top-up order.", "Promotional items ordered.",
        "Standard visit; order processed.",
    ]
    unsuccessful_notes = [
        "Shop closed at time of visit.", "Customer not available.", "Stock sufficient; no order needed.",
        "Payment overdue; order withheld.", "Customer requested reschedule.",
        "Cold-chain issue; order deferred.", "Order cancelled by customer.",
        "Credit limit exceeded.", "Competitor promotion; no order.", "No decision-maker present.",
    ]

    day_name_to_weekday = {
        "Saturday": 5, "Sunday": 6, "Monday": 0,
        "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
    }

    rows = []
    visit_counter = {}

    for _, cust in customer_df.iterrows():
        ter_id = cust["territory_id"]
        staff_pool = territory_staff.get(ter_id, [])
        if not staff_pool:
            continue

        target_weekday = day_name_to_weekday.get(cust["preferred_visit_day"])
        if target_weekday is None:
            continue

        # All candidate visit dates for this customer in the window
        cur = start_date
        while cur.weekday() != target_weekday:
            cur += timedelta(days=1)
        visit_dates = []
        while cur <= end_date:
            visit_dates.append(cur)
            cur += timedelta(days=7)

        sp_id, van_id = random.choice(staff_pool)
        tier      = cust["volume_tier"]
        lifecycle = cust.get("lifecycle_state", "Active")

        lifecycle_adj = {
            "Active": 0.00, "New": +0.05, "At Risk": -0.15,
            "Dormant": -0.30, "Churned": -0.50,
        }
        adj_prob = float(np.clip(
            success_probs[tier] + lifecycle_adj.get(lifecycle, 0), 0.05, 0.99
        ))

        mu, sigma   = amount_params[tier]
        city_code   = ter_id.replace("TER_", "")
        visit_counter.setdefault(ter_id, 0)

        for vdate in visit_dates:

            # ── Holiday check — skip if any entity is on holiday ─────────────
            if vdate in ter_blocked.get(ter_id, set()):
                continue   # whole territory closed (Friday, public holiday, etc.)
            if vdate in sp_blocked.get(sp_id, set()):
                continue   # salesperson on sick/annual leave
            if vdate in van_blocked.get(van_id, set()):
                continue   # van under maintenance

            visit_counter[ter_id] += 1
            successful = random.random() < adj_prob
            if successful:
                amount = round(float(np.clip(np.random.normal(mu, sigma), mu * 0.1, mu * 3.5)), 2)
                note   = random.choice(successful_notes)
            else:
                amount = 0.0
                note   = random.choice(unsuccessful_notes)

            rows.append({
                "visit_id":           f"VIS_{city_code}_{visit_counter[ter_id]:06d}",
                "visit_date":         vdate,
                "customer_id":        cust["customer_id"],
                "sales_id":           sp_id,
                "van_id":             van_id,
                "successful_visit":   successful,
                "transaction_amount": amount,
                "notes":              note,
            })

    visit_df = pd.DataFrame(rows)
    visit_df = visit_df.sort_values(["visit_date", "visit_id"]).reset_index(drop=True)
    return visit_df


def generate_all(seed=42, output_dir="saudi_master_data_output", write_files=True):
    set_seed(seed)

    territory_df = generate_territories()
    salesperson_df = generate_salespeople(territory_df)
    van_df = generate_vans(territory_df, salesperson_df)
    salesperson_df = assign_vans_to_salespeople(salesperson_df, van_df)
    territory_df = assign_defaults(territory_df, salesperson_df, van_df)
    customer_df = generate_customers(territory_df)
    holiday_df = generate_holidays(territory_df, salesperson_df, van_df)
    config_df = generate_config()
    # Generate visits first; then RFM consumes real visit signals
    visit_df_for_rfm = generate_visits(customer_df, salesperson_df, van_df, territory_df, holiday_df=holiday_df)
    rfm_scores_df = generate_rfm_scores(customer_df, visit_df=visit_df_for_rfm)

    validate_all(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df)
    report = data_quality_report(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df)

    outputs = {
        "territory": territory_df,
        "salesperson": salesperson_df,
        "van": van_df,
        "customer": customer_df,
        "holiday": holiday_df,
        "config": config_df,
        "rfm_scores": rfm_scores_df,
        "visit": visit_df_for_rfm,
    }

    if write_files:
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        for name, df in outputs.items():
            df.to_csv(out / f"{name}.csv", index=False)
        with open(out / "data_quality_report.json", "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, default=str, ensure_ascii=False)
        print(f"Wrote CSV/JSON outputs to: {out.resolve()}")

    return outputs, report

# Generate data
outputs, report = generate_all(seed=42)
territory_df = outputs["territory"]
salesperson_df = outputs["salesperson"]
van_df = outputs["van"]
customer_df = outputs["customer"]
holiday_df = outputs["holiday"]
config_df = outputs["config"]
rfm_scores_df = outputs["rfm_scores"]

print(json.dumps(report, indent=2, default=str, ensure_ascii=False))


Wrote CSV/JSON outputs to: D:\Data Science\Basamh\JP\journey-planner\saudi_master_data_output
{
  "row_counts": {
    "territory": 3,
    "salesperson": 9,
    "van": 12,
    "customer": 300,
    "holiday": 98,
    "config": 10,
    "rfm_scores": 300
  },
  "customer_tiers_by_territory": {
    "TER_DMM": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    },
    "TER_JED": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    },
    "TER_RUH": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    }
  },
  "lifecycle_distribution": {
    "Active": 0.65,
    "New": 0.123,
    "At Risk": 0.12,
    "Dormant": 0.103,
    "Churned": 0.003
  },
  "payment_type_distribution": {
    "cash": 0.637,
    "credit": 0.363
  },
  "cold_truck_required_share": 0.523,
  "rfm_segments": {
    "Medium": 130,
    "High": 95,
    "Low": 75
  },
  "validation_status": "passed",
  "generated_at": "2026-05-28T12:56:42"
}


In [22]:
# visit_df is generated inside generate_all (before RFM) so that RFM can
# consume real visit signals. Expose it here for inspection.
visit_df = outputs["visit"]

print(f"visit_df shape: {visit_df.shape}")
print(f"Date range   : {visit_df['visit_date'].min()}  →  {visit_df['visit_date'].max()}")
print(f"Success rate : {visit_df['successful_visit'].mean():.1%}")
print(f"Total revenue: SAR {visit_df['transaction_amount'].sum():,.0f}")
display(visit_df.head(10))

# Persist to CSV alongside the other master-data outputs
from pathlib import Path
out = Path("saudi_master_data_output")
out.mkdir(parents=True, exist_ok=True)
visit_df.to_csv(out / "visit.csv", index=False)
print(f"\nSaved → {(out / 'visit.csv').resolve()}")

visit_df shape: (14672, 8)
Date range   : 2025-06-02  →  2026-05-10
Success rate : 73.8%
Total revenue: SAR 45,700,179


,visit_id,visit_date,customer_id,sales_id,van_id,successful_visit,transaction_amount,notes
0,VIS_DMM_000196,2025-06-02,CUS_DMM_0005,SAL_DMM_002,VAN_DMM_002,True,1022.57,Promotional items ordered.
1,VIS_DMM_000391,2025-06-02,CUS_DMM_0009,SAL_DMM_003,VAN_DMM_003,True,788.18,Paid on delivery.
2,VIS_DMM_000636,2025-06-02,CUS_DMM_0014,SAL_DMM_001,VAN_DMM_001,True,818.49,Paid on delivery.
3,VIS_DMM_001223,2025-06-02,CUS_DMM_0026,SAL_DMM_001,VAN_DMM_001,True,645.03,Standard visit; order processed.
4,VIS_DMM_001370,2025-06-02,CUS_DMM_0029,SAL_DMM_003,VAN_DMM_003,True,5497.57,Invoice issued.
5,VIS_DMM_001566,2025-06-02,CUS_DMM_0033,SAL_DMM_002,VAN_DMM_002,False,0.00,Order cancelled by customer.
6,VIS_DMM_001615,2025-06-02,CUS_DMM_0034,SAL_DMM_002,VAN_DMM_002,True,2462.86,Order placed successfully.
7,VIS_DMM_001810,2025-06-02,CUS_DMM_0038,SAL_DMM_003,VAN_DMM_003,True,727.88,Invoice issued.
8,VIS_DMM_001957,2025-06-02,CUS_DMM_0041,SAL_DMM_001,VAN_DMM_001,True,12325.11,Promotional items ordered.
9,VIS_DMM_002006,2025-06-02,CUS_DMM_0042,SAL_DMM_002,VAN_DMM_002,True,3663.03,Paid on delivery.



Saved → D:\Data Science\Basamh\JP\journey-planner\saudi_master_data_output\visit.csv


In [23]:
# ----- Visit table diagnostics -----
print("Visits per territory")
display(
    visit_df.merge(customer_df[["customer_id", "territory_id"]], on="customer_id")
    .groupby("territory_id")
    .agg(total_visits=("visit_id", "count"),
         successful=("successful_visit", "sum"),
         success_rate=("successful_visit", "mean"),
         total_revenue=("transaction_amount", "sum"))
    .round({"success_rate": 3, "total_revenue": 2})
)

print("\nVisits per salesperson (top 10 by revenue)")
display(
    visit_df.groupby("sales_id")
    .agg(visits=("visit_id", "count"),
         success_rate=("successful_visit", "mean"),
         revenue=("transaction_amount", "sum"))
    .sort_values("revenue", ascending=False)
    .head(10)
    .round({"success_rate": 3, "revenue": 2})
)

Visits per territory


,total_visits,successful,success_rate,total_revenue
territory_id,,,,
TER_DMM,4890,3705,0.758,15615153.70
TER_JED,4890,3588,0.734,14418964.96
TER_RUH,4892,3537,0.723,15666060.34



Visits per salesperson (top 10 by revenue)


,visits,success_rate,revenue
sales_id,,,
SAL_DMM_001,1715,0.748,6340719.13
SAL_JED_001,1807,0.745,6063796.41
SAL_RUH_003,1862,0.743,5835330.43
SAL_DMM_003,1659,0.780,5317007.53
SAL_RUH_002,1421,0.716,5200555.15
SAL_JED_002,1813,0.743,4883795.82
SAL_RUH_001,1609,0.705,4630174.76
SAL_DMM_002,1516,0.744,3957427.04
SAL_JED_003,1270,0.704,3471372.73



## Preview generated tables

Run the cells below after generation to inspect samples and distribution checks.


In [24]:

print("Territories")
display(territory_df)

print("Salespeople")
display(salesperson_df.head(10))

print("Vans")
display(van_df.head(10))

print("Customers")
display(customer_df.head(10))

print("RFM scores")
display(rfm_scores_df.head(10))


Territories


,territory_id,territory_name,warehouse_lat,warehouse_lng,warehouse_address,center_lat,center_lng,radius_km,default_salesperson,default_van
0,TER_RUH,Riyadh Central,24.5790,46.8237,"Industrial Area, Riyadh",24.7136,46.6753,25,SAL_RUH_001,VAN_RUH_001
1,TER_JED,Jeddah North,21.3429,39.2357,"Al Khomrah Logistics Area, Jeddah",21.5433,39.1728,22,SAL_JED_002,VAN_JED_002
2,TER_DMM,Dammam Metro,26.2926,50.1629,"2nd Industrial City, Dammam",26.4207,50.0888,20,SAL_DMM_002,VAN_DMM_002


Salespeople


,sales_id,name,territory_id,working_hours_per_day,shift_start_time,assigned_van,active_status,performance_multiplier
0,SAL_RUH_001,Arif Rahman,TER_RUH,8.0,09:00,VAN_RUH_001,True,1.04
1,SAL_RUH_002,Nasser Al-Dossari,TER_RUH,8.0,09:00,VAN_RUH_002,True,0.99
2,SAL_RUH_003,Abdullah Al-Qahtani,TER_RUH,8.0,09:00,VAN_RUH_003,True,0.97
3,SAL_JED_001,Ahmed Al-Anazi,TER_JED,8.0,09:00,VAN_JED_001,True,0.98
4,SAL_JED_002,Majed Al-Shammari,TER_JED,8.0,09:00,VAN_JED_002,True,1.14
5,SAL_JED_003,Imran Ahmed,TER_JED,8.0,09:00,VAN_JED_003,True,0.98
6,SAL_DMM_001,Khalid Al-Ghamdi,TER_DMM,8.0,09:00,VAN_DMM_001,True,0.97
7,SAL_DMM_002,Hassan Al-Yami,TER_DMM,8.0,09:00,VAN_DMM_002,True,1.13
8,SAL_DMM_003,Fahad Al-Otaibi,TER_DMM,8.0,09:00,VAN_DMM_003,True,0.88


Vans


,van_id,registration_no,cold_truck_enabled,territory_id,active_status
0,VAN_RUH_001,RUG 2139,True,TER_RUH,True
1,VAN_RUH_002,RUJ 2307,False,TER_RUH,True
2,VAN_RUH_003,RUM 5554,True,TER_RUH,True
3,VAN_RUH_004,RUF 7065,True,TER_RUH,True
4,VAN_JED_001,JEW 2169,False,TER_JED,True
5,VAN_JED_002,JEX 5010,True,TER_JED,True
6,VAN_JED_003,JEU 4598,False,TER_JED,True
7,VAN_JED_004,JEY 1916,True,TER_JED,True
8,VAN_DMM_001,DMK 7572,True,TER_DMM,True
9,VAN_DMM_002,DMS 6155,True,TER_DMM,True


Customers


,customer_id,shop_name,gps_lat,gps_lng,locality,territory_id,customer_rating,review_rating,shop_category,cold_truck_required,volume_tier,payment_type,credit_limit,outstanding_balance,lifecycle_state,acquisition_date,preferred_visit_day,preferred_order_window
0,CUS_RUH_0001,Al Waha Fresh Mini Market,24.769194,46.600485,Hittin,TER_RUH,2,4.4,Mini Market,False,LOW,cash,0.00,0.00,Active,2025-02-27,Tuesday,Morning
1,CUS_RUH_0002,Al Othman Banquet Kitchen,24.780751,46.599296,Hittin,TER_RUH,3,3.9,Catering Kitchen,True,MED,cash,0.00,0.00,Active,2023-05-24,Tuesday,Midday
2,CUS_RUH_0003,Al Qassim Al Yasmin Grocery,24.844884,46.639722,Al Yasmin,TER_RUH,5,3.7,Grocery,False,MED,cash,0.00,0.00,At Risk,2022-10-20,Sunday,Morning
3,CUS_RUH_0004,Al Waha Fresh Meat,24.769710,46.596453,Hittin,TER_RUH,2,3.7,Butchery,True,HIGH,credit,71636.53,26962.89,Active,2022-09-25,Sunday,Midday
4,CUS_RUH_0005,Al Manar Fresh Butchery,24.829825,46.606460,Al Yasmin,TER_RUH,3,3.1,Butchery,True,MED,cash,0.00,0.00,Active,2023-01-03,Tuesday,Afternoon
5,CUS_RUH_0006,Al Shammari Chilled Foods,24.757268,46.589639,Hittin,TER_RUH,2,4.2,Cold Store,True,LOW,credit,5857.20,3237.35,New,2022-06-13,Saturday,Afternoon
6,CUS_RUH_0007,Olaya Grill,24.701275,46.657092,Olaya,TER_RUH,5,4.8,Restaurant,True,LOW,cash,0.00,0.00,New,2025-02-03,Wednesday,Afternoon
7,CUS_RUH_0008,Hittin Cafe,24.761057,46.603037,Hittin,TER_RUH,3,3.2,Cafe,True,MED,credit,35066.71,4264.31,Active,2022-08-08,Wednesday,Midday
8,CUS_RUH_0009,Al Nada Fresh Mini Market,24.661470,46.736475,Al Malaz,TER_RUH,3,3.4,Mini Market,False,LOW,cash,0.00,0.00,Dormant,2021-06-14,Thursday,Afternoon
9,CUS_RUH_0010,Al Ghamdi Mini Market,24.705431,46.703755,Al Sulaymaniyah,TER_RUH,2,3.8,Mini Market,False,MED,cash,0.00,0.00,Dormant,2025-06-06,Monday,Morning


RFM scores


,customer_id,territory_id,last_successful_visit,recency,frequency,monetary,r_score,f_score,m_score,rfm_combined,seasonality_score,territory_score,locality_score,rating_score,final_customer_score,rfm_segment_final,customer_rank
0,CUS_DMM_0003,TER_DMM,2026-05-10,18.0,2.0,32094.13,1.000000,1.0,1.000000,1.000000,1.000000,0.477316,0.734222,0.75,0.896154,High,1
1,CUS_DMM_0012,TER_DMM,2026-05-10,18.0,2.0,27144.44,1.000000,1.0,0.845776,0.948592,1.000000,0.477316,0.930638,0.75,0.895232,High,2
2,CUS_DMM_0077,TER_DMM,2026-05-07,21.0,2.0,22012.66,0.750000,1.0,0.685878,0.811959,0.986755,0.477316,1.000000,1.00,0.868542,High,3
3,CUS_DMM_0050,TER_DMM,2026-05-10,18.0,2.0,19153.44,1.000000,1.0,0.596790,0.865597,1.000000,0.477316,0.930638,0.75,0.862034,High,4
4,CUS_DMM_0093,TER_DMM,2026-05-09,19.0,2.0,23637.17,0.916667,1.0,0.736495,0.884387,1.000000,0.477316,0.540430,1.00,0.855530,High,5
5,CUS_DMM_0039,TER_DMM,2026-05-06,22.0,2.0,30977.87,0.666667,1.0,0.965219,0.877295,0.986755,0.477316,0.734222,0.75,0.843098,High,6
6,CUS_DMM_0053,TER_DMM,2026-05-09,19.0,2.0,6694.97,0.916667,1.0,0.208604,0.708424,1.000000,0.477316,1.000000,1.00,0.831101,High,7
7,CUS_DMM_0010,TER_DMM,2026-05-09,19.0,2.0,13494.66,0.916667,1.0,0.420471,0.779046,1.000000,0.477316,0.930638,0.75,0.827414,High,8
8,CUS_DMM_0052,TER_DMM,2026-05-07,21.0,2.0,18224.77,0.750000,1.0,0.567854,0.772618,0.986755,0.477316,0.930638,0.75,0.820869,High,9
9,CUS_DMM_0067,TER_DMM,2026-05-10,18.0,2.0,18317.25,1.000000,1.0,0.570735,0.856912,1.000000,0.477316,0.540430,0.75,0.819539,High,10


In [42]:
customer_df.merge(salesperson_df, how= "inner",  on= 'territory_id').to_excel('customer.xlsx')

In [35]:
display(salesperson_df[["sales_id", "assigned_van", "cold_capable"]])

print()

print(customer_df[['customer_id', 'cold_truck_required']])

,sales_id,assigned_van,cold_capable
0,SAL_RUH_001,VAN_RUH_001,True
1,SAL_RUH_002,VAN_RUH_002,False
2,SAL_RUH_003,VAN_RUH_003,True
3,SAL_JED_001,VAN_JED_001,False
4,SAL_JED_002,VAN_JED_002,True
5,SAL_JED_003,VAN_JED_003,False
6,SAL_DMM_001,VAN_DMM_001,True
7,SAL_DMM_002,VAN_DMM_002,True
8,SAL_DMM_003,VAN_DMM_003,True



      customer_id  cold_truck_required
0    CUS_RUH_0001                False
1    CUS_RUH_0002                 True
2    CUS_RUH_0003                False
3    CUS_RUH_0004                 True
4    CUS_RUH_0005                 True
..            ...                  ...
295  CUS_DMM_0096                 True
296  CUS_DMM_0097                False
297  CUS_DMM_0098                False
298  CUS_DMM_0099                 True
299  CUS_DMM_0100                False

[300 rows x 2 columns]


In [25]:
from saudi_multi_salesperson_scheduler_new import MultiSalespersonScheduler

result = MultiSalespersonScheduler().create_schedules(
    customer_df=customer_df,
    rfm_scores_df=rfm_scores_df,
    salesperson_df=salesperson_df,
    holiday_df=holiday_df,
    territory_df=territory_df,
    config_df=config_df,
    van_df=van_df,
    month_start_date="2026-06-01",
)


Territory TER_RUH: assigning 100 customers to 3 salespeople via KMeans clustering...
  SAL_RUH_001 → 42 customers  {'Medium': 15, 'High': 14, 'Low': 13}
  SAL_RUH_002 → 23 customers  {'High': 14, 'Medium': 7, 'Low': 2}
  SAL_RUH_003 → 35 customers  {'High': 14, 'Medium': 12, 'Low': 9}

  Scheduling SAL_RUH_001 (42 customers, 16 valid days)...
  [SAL_RUH_001] FEASIBLE | obj=81405775

  Scheduling SAL_RUH_002 (23 customers, 16 valid days)...
  [SAL_RUH_002] OPTIMAL | obj=65368460

  Scheduling SAL_RUH_003 (35 customers, 16 valid days)...
  [SAL_RUH_003] OPTIMAL | obj=48859450

Territory TER_JED: assigning 100 customers to 3 salespeople via KMeans clustering...
  SAL_JED_001 → 23 customers  {'High': 8, 'Medium': 8, 'Low': 7}
  SAL_JED_002 → 56 customers  {'Medium': 33, 'Low': 15, 'High': 8}
  SAL_JED_003 → 21 customers  {'Low': 10, 'High': 7, 'Medium': 4}

  Scheduling SAL_JED_001 (23 customers, 16 valid days)...
  [SAL_JED_001] OPTIMAL | obj=55947519

  Scheduling SAL_JED_002 (56 custom

In [26]:
from saudi_multi_salesperson_scheduler_new import build_route_map_for_salesperson, build_territory_day_map
# m = build_route_map_for_salesperson(
#     daily_schedule    = result.daily_schedule,
#     detailed_schedule = result.detailed_schedule,
#     sales_id          = "SAL_DMM_001",
#     schedule_date     = "2026-07-02",
# )
# display(m)

# m = build_route_map_for_salesperson(
#     daily_schedule    = result.daily_schedule,
#     detailed_schedule = result.detailed_schedule,
#     sales_id          = "SAL_DMM_002",
#     schedule_date     = "2026-07-02",
# )
# display(m)

# m = build_route_map_for_salesperson(
#     daily_schedule    = result.daily_schedule,
#     detailed_schedule = result.detailed_schedule,
#     sales_id          = "SAL_DMM_003",
#     schedule_date     = "2026-07-02",
# )
# display(m)

wh = result.territory_warehouses["TER_JED"]
m = build_route_map_for_salesperson(
    daily_schedule    = result.daily_schedule,
    detailed_schedule = result.detailed_schedule,
    sales_id          = "SAL_JED_001",
    schedule_date     = "2026-06-16",
    warehouse_lat     = wh[0],
    warehouse_lng     = wh[1],
)
display(m)

# ALL salespeople in a territory on one map
m = build_territory_day_map(
    result        = result,
    territory_id  = "TER_JED",
    schedule_date = "2026-06-16",
)
m

## Saving Monthly Plan

In [27]:
import pandas as pd

# Define the output file name
excel_file_path = "monthly_plan_new.xlsx"

with pd.ExcelWriter(excel_file_path, engine="openpyxl") as writer:
    # 1. Save the main aggregated dataframes to their own sheets
    result.detailed_schedule.to_excel(
        writer, sheet_name="Detailed Schedule", index=False
    )
    result.daily_schedule.to_excel(writer, sheet_name="Daily Schedule", index=False)

    # 2. Loop through the salesperson dictionary and write each to a sheet
    for name, sp_result in result.salesperson_results.items():
        # Excel sheet names cannot exceed 31 characters
        safe_sheet_name = f"SP_{name}"

        # Check if the salesperson result object itself has dataframes to unpack
        # If it is a dictionary, use: sp_result_dict = sp_result
        # If it is another dataclass, convert it to a dict to read its attributes
        if hasattr(sp_result, "__dataclass_fields__"):
            from dataclasses import asdict

            sp_data = asdict(sp_result)
        elif isinstance(sp_result, dict):
            sp_data = sp_result
        else:
            # Fallback: tries to extract internal attributes if it's a custom class
            sp_data = vars(sp_result)

        # Look for any pandas DataFrames inside the salesperson's data to save
        for key, value in sp_data.items():
            if isinstance(value, pd.DataFrame):
                # Creates a unique sheet name like "SP_John_detailed"
                sub_sheet_name = f"SP_{name}_{key}"
                value.to_excel(writer, sheet_name=sub_sheet_name, index=False)

print(f"Successfully saved results to {excel_file_path}")


d:\Data Science\Basamh\JP\journey-planner\.venv\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Successfully saved results to monthly_plan_new.xlsx


## Test Case

In [28]:
# Now validate
from validate_schedule import validate_schedule

validation_report = validate_schedule(
    result=result,
    customer_df=customer_df,
    salesperson_df=salesperson_df,
    van_df=van_df,
    territory_df=territory_df,
    holiday_df=holiday_df,
    config_df=config_df,
    rfm_scores_df=rfm_scores_df,
    month_start="2026-06-01"
)


❌ Cold truck requirement – 64 violation(s):
   Customer CUS_JED_0083 requires cold truck but assigned to SAL_JED_001 (van not cold-capable)
   Customer CUS_JED_0067 requires cold truck but assigned to SAL_JED_001 (van not cold-capable)
   Customer CUS_JED_0004 requires cold truck but assigned to SAL_JED_001 (van not cold-capable)
   Customer CUS_JED_0006 requires cold truck but assigned to SAL_JED_001 (van not cold-capable)
   Customer CUS_JED_0034 requires cold truck but assigned to SAL_JED_001 (van not cold-capable)
   ... and 59 more

❌ Min 1 visit per active customer – 49 violation(s):
   Active customer CUS_JED_0008 has 0 visits
   Active customer CUS_JED_0099 has 0 visits
   Active customer CUS_JED_0071 has 0 visits
   Active customer CUS_JED_0057 has 0 visits
   Active customer CUS_JED_0092 has 0 visits
   ... and 44 more
✅ Max visits per month by segment – OK
✅ Daily customer count limit – OK
✅ Daily total time limit – OK
✅ Holiday blocking – OK
✅ Visit minutes (should be 22) 

In [29]:

# print("Tier counts per territory")
# display(customer_df.groupby(["territory_id", "volume_tier"]).size().unstack(fill_value=0))

# print("Lifecycle distribution")
# display(customer_df["lifecycle_state"].value_counts(normalize=True).rename("share").round(3))

# print("RFM segments")
# display(rfm_scores_df["segment"].value_counts())

# print("Output files")
# for path in sorted(Path("saudi_master_data_output").glob("*")):
#     print(path)



## Notes

- This notebook intentionally excludes route, visit, and basket-level transaction tables.
- The holiday dates are synthetic placeholders suitable for master-data demos. For production Saudi holiday calendars, replace those placeholder rows with an official calendar source.
- `generate_all(seed=42)` is deterministic for repeatable demos. Change the seed to create a different synthetic dataset.
